# Занятие 8. Методы полей сил

## 💡 Задание 1. Метадинамика дипептида с пролином

В занятии мы строили поверхность свободной энергии F(φ, ψ) дипептида аланина. Замените второй остаток на пролин: в tleap последовательность `NALA CPRO` вместо `NALA CALA`. Используйте тот же протокол: pdb2gmx, бокс, вода, минимизация, NVT, NPT, затем well-tempered метадинамика 5 нс по φ и ψ с теми же параметрами холмов, что в разделе 1.3.2.

**Что нужно сделать:**
1. Собрать систему и запустить метадинамическую симуляцию. Проверьте по `COLVAR`, что обе CV считаются по правильным атомам: φ = C₁–N₂–Cα₂–C₂, ψ = N₁–Cα₁–C₁–N₂.
2. Построить F(φ, ψ) через `sum_hills` и нарисовать карту в том же стиле, что в занятии, вместе с φ(t) и ψ(t).
3. Сравнить  поверхность свободной энергии F(φ, ψ) с картой Ala-Ala из занятия.

**Вопросы:**
- Какие минимумы остались и какова разность свободных энергий между ними?
- Нужны ли для этой молекулы обе CV? Что бы вы выбрали, если бы разрешена была только одна?


## ⚕️Задание 2. Пролин в спирали

В занятии поли-аланин сворачивался в α-спираль из вытянутой цепи. Теперь пойдём с другой стороны: построим готовую прямую спираль и посмотрим, что сделает с ней метод Монте-Карло с критерием Метрополиса, если в середину вставить пролин. Две последовательности по 20 остатков: `AAAAAAAAAAAAAAAAAAAA` и `AAAAAAAAAPAAAAAAAAAA` (пролин на десятом месте). Спирали строит tleap командой `impose`, как стартовые конформации в разделе 1.1; ячейка ниже строит обе структуры, загружает их в PyRosetta и переупаковывает боковые цепи.

**Что нужно сделать:**
1. Запустить ячейку сборки и сравнить энергии двух спиралей до переупаковки. 
2. Запустить из обеих спиралей Монте-Карло как в занятии: kT = 0.5 REU, шаг ±60°, 160 000 шагов.
3. Нарисовать E, RMSD по ходу Монте-Карло.

**Вопросы:**
- Что происходит с энергией спирали с пролином в первые две тысячи попыток? Стоит ли этот участок включать в средние?
- Как ведёт себя угол излома у спирали с пролином и у поли-аланина? Остаётся ли спираль спиралью по обе стороны от пролина?
- Почему пролин ломает спираль?

In [ ]:
# Две α-спирали по 20 остатков: tleap строит цепь с φ = -57°, ψ = -47° у всех остатков диапазона { { 1 20 } },
# PyRosetta читает PDB и переупаковывает боковые цепи (кольцо пролина в tleap собрано не так, как любит ref2015,
# а МК по φ и ψ его не трогает)
import os
from pyrosetta import pose_from_file, get_fa_scorefxn
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover

SEQS = {"ala20": "A" * 20, "pro20": "A" * 9 + "P" + "A" * 10}
AA3 = {"A": "ALA", "P": "PRO"}
sf = get_fa_scorefxn()                                     # ref2015, энергия в REU

helix = {}
for name, seq in SEQS.items():
    res = [AA3[a] for a in seq]
    res[0], res[-1] = "N" + res[0], "C" + res[-1]          # заряженные концевые остатки, как в разделе 1.1
    n = len(res)
    tleap_in = f"""\
source leaprc.protein.ff14SB
pep = sequence {{ {" ".join(res)} }}
impose pep {{ {{ 1 {n} }} }} {{ {{ "N" "CA" "C" "N" -47.0 }} {{ "C" "N" "CA" "C" -57.0 }} }}
savepdb pep {name}_helix.pdb
quit
"""
    with open(f"{name}_tleap.in", "w") as f:
        f.write(tleap_in)
    os.system(f"tleap -f {name}_tleap.in > {name}_tleap.log 2>&1")

    pose = pose_from_file(f"{name}_helix.pdb")
    E0 = sf(pose)
    task = pyrosetta.standard_packer_task(pose)
    task.restrict_to_repacking()                           # только ротамеры, последовательность не трогаем
    PackRotamersMover(sf, task).apply(pose)
    helix[name] = pose
    print(f"{name}: {seq}  E = {E0:7.1f} REU из tleap, {sf(pose):7.1f} REU после переупаковки")

## 💊 Задание 3. Докинг 2-нафтамидина

В занятии мы проводили докинг бензамидина в карман трипсина. Повторите расчёт для 2-нафтамидина: та же мишень, тот же протокол, другой лиганд:

| Лиганд | SMILES | $K_i$, мкМ |
|---|---|---|
| Бензамидин | `NC(=[NH2+])c1ccccc1` | 18.4 |
| 2-Нафтамидин | `NC(=[NH2+])c1ccc2ccccc2c1` | 6.6 |

**Что нужно сделать:**
1. Скопировать разделы 3.2.1–3.2.3 из занятия. Заменить SMILES и трёхбуквенное имя остатка (например, `NAF`). Обратите внимание на протонированную форму: как и бензамидин, нафтамидин в воде заряжен положительно.
2. Запустить 100 независимых решений из случайных стартовых поз в том же центре кармана.
3. Нарисовать десять лучших и десять худших решений вместе с белком.

**Вопросы:**
- Как расположено нафталиновое кольцо в кармане? С какими остатками оно контактирует?
- Сравните ΔE лучшего решения с бензамидином из занятия. Какой лиганд Rosetta считает более сильным?
- Насколько разброс ΔE среди десяти лучших решений для одного лиганда больше или меньше разности между лигандами? Что это говорит о надёжности такого сравнения?

* Константы ингибирования: бензамидин ([Mares-Guia, Shaw, 1965](https://doi.org/10.1016/S0021-9258(18)97474-0)), 2-нафтамидин ([Tanizawa, Ishii, Kanaoka, 1971](https://academic.oup.com/jb/article-abstract/69/5/893/2184754))